# 01. Interactive Fiction, Visual Novels & Narrative RPGs

**Preset:** `InteractiveFictionPreset`

## Why `canonarc` Fits This Domain

Traditional LLM memory frameworks (`mem0`, `zep`, `langmem`) assume a single linear, wall-clock chat log. Interactive fiction, visual novels (Ren'Py, Ink), and narrative RPGs break every one of those assumptions:

1. **Discrete Causal Steps:** Time advances in turns, scenes, or beats (`step: int`), not real-world timestamps.
2. **Save-Scumming & Route Branching:** Players constantly reload earlier saves or explore alternate story branches. `canonarc` provides **zero-copy branching** (`branch()`) and **`< 1ms` atomic rollback** (`rollback_to()`) over a single portable `.sqlite` save file.
3. **Dramatic Irony & Anti-Metagaming:** Characters should only react to events they witnessed or were told about, while the narrator still sees who is ignorant of a secret (`| ignorant: ...`).
4. **Voice Consistency Without Drift:** The Tier 5 **Gated Voice Bank** prevents generic LLM lines from polluting future few-shot dialogue prompts.

In this notebook, we build a branching dark-fantasy RPG scene from scratch and walk through Tiers 1–6, anti-drift voice gating, `step_transaction()`, branching, timeline diffing, rollback, and save-slot export.

In [1]:
import tempfile
from pathlib import Path

from canonarc import (
    EdgeDelta,
    EntityDelta,
    EntityRecord,
    EpisodeRecord,
    InteractiveFictionPreset,
    CanonArcEngine,
    StepDelta,
    VoiceExemplar,
)

# Initialize an in-memory engine with the default InteractiveFictionPreset
engine = CanonArcEngine(":memory:", preset=InteractiveFictionPreset())
print("Engine initialized on timeline:", engine.active_timeline)

Engine initialized on timeline: main


## Step 1: Seed Tier 1 Entities, Aliases & Initial Tier 2 Stances

- **Tier 1 (`base_attributes`)** holds stable character metadata (title, gender, archetype).
- **Tier 4 (`aliases`)** enables 1-hop coreference so prose mentioning `"the old wizard"` automatically resolves to `char_merlin`.
- **Tier 2 (`edges`)** stores **asymmetric directed stances** (`Arthur -> Merlin` is independent of `Merlin -> Arthur`).

In [2]:
# Step 0: Register cast and aliases
engine.register_entity(
    EntityRecord(
        entity_id="char_arthur",
        canonical_name="Arthur",
        base_attributes={"title": "King", "gender": "M", "archetype": "noble_ruler"},
        state={"mood": "calm", "location_id": "loc_council_chamber"},
    ),
    aliases=["the King", "His Majesty"],
    step=0,
)
engine.register_entity(
    EntityRecord(
        entity_id="char_merlin",
        canonical_name="Merlin",
        base_attributes={"title": "Court Wizard", "gender": "M", "archetype": "enigmatic_mentor"},
        state={"mood": "composed", "location_id": "loc_council_chamber"},
    ),
    aliases=["the old wizard", "the Court Wizard"],
    step=0,
)
engine.register_entity(
    EntityRecord(
        entity_id="char_mordred",
        canonical_name="Mordred",
        base_attributes={"title": "Knight", "gender": "M", "archetype": "ambitious_heir"},
        state={"mood": "watchful", "location_id": "loc_courtyard"},
    ),
    step=0,
)

# Step 10: Establish initial formal, trusting stances
engine.commit_step(
    step=10,
    delta=StepDelta(
        edges=[
            EdgeDelta(
                source_id="char_arthur",
                target_id="char_merlin",
                state={"register": "formal", "tone": "respectful", "trust": 0.9},
            ),
            EdgeDelta(
                source_id="char_merlin",
                target_id="char_arthur",
                state={"register": "formal", "tone": "patient", "trust": 0.85},
            ),
        ]
    ),
)
print("Seeded 3 characters and initial stances at Step 10.")

Seeded 3 characters and initial stances at Step 10.


## Step 2: Tier 5 Gated Voice Bank (Anti-Drift Protection)

If you feed every LLM-generated dialogue line back into memory, a single bland line gets retrieved later as a few-shot example and gradually flattens every character's voice (**error compounding**).

`canonarc` runs every candidate line through a **2-stage `IngestionGate`**:
1. **Deterministic Lexical Pre-Gate:** Checks word-count bounds and lexical richness (Root-TTR).
2. **Quality Score Gate:** Requires `quality_score >= 8.0` before indexing.

Let's see the gate reject a bland/low-quality line while accepting rich in-character anchors for both `formal` and `informal` registers.

In [3]:
candidates = [
    # Should be REJECTED (too short & low quality_score)
    VoiceExemplar(
        step=15,
        speaker_id="char_arthur",
        text="Yes, okay.",
        metadata={"register": "formal", "tone": "respectful"},
        quality_score=4.5,
    ),
    # Should be ACCEPTED (formal, respectful)
    VoiceExemplar(
        step=15,
        speaker_id="char_arthur",
        addressee_id="char_merlin",
        text="I beseech your counsel, venerable wizard, for the realm stands upon a perilous precipice.",
        metadata={"register": "formal", "tone": "respectful"},
        quality_score=9.4,
    ),
    # Should be ACCEPTED (informal, furious — for later conflict scenes)
    VoiceExemplar(
        step=82,
        speaker_id="char_arthur",
        addressee_id="char_merlin",
        text="Spare me your riddles, old man—you hid the dragon prophecy while my knights bled in the ash!",
        metadata={"register": "informal", "tone": "furious"},
        quality_score=9.2,
    ),
]

accepted = engine.index_voice_exemplars(step=15, exemplars=candidates[:2])
print(f"Step 15: {len(accepted)} of 2 exemplars passed the anti-drift gate.")
assert len(accepted) == 1

Step 15: 1 of 2 exemplars passed the anti-drift gate.


## Step 3: Commit a Plot Turning Point (Tier 3 Delta + Tier 6 Episodic Ledger)

At **Step 82**, Arthur discovers Merlin concealed a prophecy. Instead of stuffing the whole event description into Arthur's `state` dict, we:
1. Record a discrete **Tier 6 `EpisodeRecord`** (`ep_step82_prophecy_dispute`) with explicit `participants` and `witnesses` (`char_mordred` is outside in the courtyard, so he is **not** a witness).
2. Link Arthur's mood shift and the broken `Arthur -> Merlin` edge to that episode via **`caused_by_episode_id`** and **`reason`**.

Because `canonarc` uses SQLite window functions (`LEAD(...) OVER`), Arthur's previous mood (`calm` at Step 0) is automatically preserved as `prior_state` without an extra query.

In [4]:
engine.commit_step(
    step=82,
    delta=StepDelta(
        entities=[
            EntityDelta(
                entity_id="char_arthur",
                state={"mood": "furious"},
                caused_by_episode_id="ep_step82_prophecy_dispute",
                reason="Discovered Merlin concealed the dragon prophecy",
            ),
            # Mordred enters the council chamber AFTER the argument ends
            EntityDelta(
                entity_id="char_mordred",
                state={"location_id": "loc_council_chamber"},
            ),
        ],
        edges=[
            EdgeDelta(
                source_id="char_arthur",
                target_id="char_merlin",
                state={"register": "informal", "tone": "furious", "trust": -0.6},
                caused_by_episode_id="ep_step82_prophecy_dispute",
                reason="Betrayed over the concealed prophecy",
            )
        ],
        episodes=[
            EpisodeRecord(
                episode_id="ep_step82_prophecy_dispute",
                predicate="argument",
                summary="Arthur confronted Merlin in the council chamber over concealing the dragon prophecy.",
                participants={"char_arthur": "actor", "char_merlin": "target"},
                witnesses=["char_arthur", "char_merlin"],
                location_id="loc_council_chamber",
                salience=9.0,
            )
        ],
    ),
    exemplars=[candidates[2]],
)
print("Committed Step 82 confrontation, causal edge shift, and furious voice anchor.")

Committed Step 82 confrontation, causal edge shift, and furious voice anchor.


## Step 4: Unified Tier 1–6 Retrieval & Prompt Formatting (Step 85)

Now we retrieve context for Step 85 using a raw scene cue that mentions `"the old wizard"` instead of `"Merlin"`. Notice what `retrieve_context()` produces in a single pass:
- **1-hop coreference:** `"the old wizard"` resolves to `char_merlin`.
- **Trajectory:** Arthur's mood shows `(was: calm at step 0)`.
- **Causal provenance & pinning:** The `ep_step82_prophecy_dispute` episode is automatically pinned because active state/edges point to it.
- **Dramatic irony:** Because Mordred is now co-present in the scene but did not witness Step 82, the preset tags `| ignorant: char_mordred`.
- **Register-filtered voice bank:** Arthur's `informal, furious` voice exemplar is selected over his earlier `formal` line because his active edge toward Merlin is now `informal`.

In [5]:
scene_cue = 'Arthur glared at the old wizard while Mordred waited by the doors. "Speak truth now!"'
ctx = engine.retrieve_context(
    step=85,
    text=scene_cue,
    top_k_voice=2,
    top_k_episodes=3,
)

print("Resolved mentions:", [(m.surface_text, m.entity_id, m.matched_via) for m in ctx.resolved_mentions])
print()
print(engine.format_context(ctx))

Resolved mentions: [('the old wizard', 'char_merlin', 'alias'), ('mordred', 'char_mordred', 'canonical'), ('arthur', 'char_arthur', 'canonical')]

### Active Scene Narrative Context (Step: 85)

#### Characters Present & State:
- **Arthur** (char_arthur): mood: furious (was: calm at step 0) [reason: Discovered Merlin concealed the dragon prophecy]
- **Merlin** (char_merlin): mood: composed
- **Mordred** (char_mordred): mood: watchful

#### Interpersonal Stances & Register:
- char_arthur → char_merlin: register=informal, tone=furious, trust=-0.6 (because: Betrayed over the concealed prophecy)
- char_merlin → char_arthur: register=formal, tone=patient, trust=0.85

#### Episodic Memory (Relevant Past Events):
- [Step 82 | argument @ loc_council_chamber | known to: char_arthur, char_merlin | ignorant: char_mordred] Arthur confronted Merlin in the council chamber over concealing the dragon prophecy.

#### Voice Bank (Historical Dialogue Anchors):
- char_arthur [informal, furious]: "Spare me 

## Step 5: Causal Time Travel — Zero-Copy Branching, Diffing, Rollback & Save Slots

Suppose the player reloads a decision point at **Step 50** (before the Step 82 argument) to pursue a `"reconciliation_route"`:
- `engine.branch(at_step=50, name="reconciliation_route")` inserts **one metadata row**—zero nodes, edges, or vectors are copied.
- The new branch inherits all state up to Step 50 and never sees Step 82 from `main`.
- `diff_timelines()` compares any two branches at any step.
- `export_snapshot()` writes a flattened, standalone `.sqlite` save slot file.

In [6]:
# 1. Fork an alternate route at Step 50 (before the Step 82 argument)
engine.branch(at_step=50, name="reconciliation_route", switch=True)

# On the new branch, Merlin proactively shares the prophecy at Step 60
with engine.step_transaction(step=60, text="Merlin reveals the prophecy to Arthur.") as tx:
    tx.commit(
        delta=StepDelta(
            edges=[
                EdgeDelta(
                    source_id="char_arthur",
                    target_id="char_merlin",
                    state={"register": "formal", "tone": "grateful", "trust": 1.0},
                    caused_by_episode_id="ep_step60_honest_counsel",
                    reason="Merlin shared the prophecy voluntarily",
                )
            ],
            episodes=[
                EpisodeRecord(
                    episode_id="ep_step60_honest_counsel",
                    predicate="revelation",
                    summary="Merlin voluntarily revealed the dragon prophecy to Arthur before battle.",
                    participants={"char_merlin": "actor", "char_arthur": "target"},
                    witnesses=["char_merlin", "char_arthur"],
                    salience=8.5,
                )
            ],
        )
    )

# Verify branch isolation at Step 85
branch_ctx = engine.retrieve_context(step=85, mentions=["Arthur", "Merlin"])
arthur_to_merlin = [e for e in branch_ctx.active_edges if e.source_id == "char_arthur"][0]
print("reconciliation_route Arthur->Merlin:", arthur_to_merlin.state)
assert arthur_to_merlin.state["register"] == "formal"
assert arthur_to_merlin.state["tone"] == "grateful"

# 2. Diff 'main' vs 'reconciliation_route' at Step 85
diff = engine.diff_timelines("main", "reconciliation_route", at_step=85)
print("Changed records between branches:", [item["subject_key"] for item in diff["changed"]])

# 3. Export a standalone save-slot .sqlite file
with tempfile.TemporaryDirectory() as tmpdir:
    slot_path = engine.export_snapshot(Path(tmpdir) / "save_slot_01.sqlite", at_step=85)
    print("Exported standalone save slot:", slot_path.name, f"({slot_path.stat().st_size} bytes)")

# 4. Atomic < 1ms rollback on 'main' back to Step 50
deleted = engine.rollback_to(step=50, timeline_id="main")
print(f"Rolled back 'main' to Step 50 (atomically deleted {deleted} future rows).")

engine.close()

reconciliation_route Arthur->Merlin: {'register': 'formal', 'tone': 'grateful', 'trust': 1.0}
Changed records between branches: ['char_arthur', 'char_arthur:stance->char_merlin', 'char_mordred']
Exported standalone save slot: save_slot_01.sqlite (98304 bytes)
Rolled back 'main' to Step 50 (atomically deleted 7 future rows).
